# Đánh giá trên tập Validation (15%) đối với cấu hình C30

## Kết quả tuning trên train

Bước trước đã tuning cấu hình C30 bằng ba fold trên train. Notebook này giữ nguyên tham số đã chọn cho Random Forest và XGBoost, fit lại trên toàn bộ cohort train rồi đánh giá cùng các giờ validation đủ history. Không tìm tham số lại và không sử dụng test.

## Fit lại sau tuning trên Random Forest 

Lấy tham số đã chọn, học lại trên toàn bộ 22.503 giờ train, rồi dự đoán 6.050 giờ validation. Không tìm tham số lại

### 1. Đọc lại kết quả tuning đã chốt trước đó

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

project_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "data/preprocessed/hourly_unfilled.csv").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import tuning_cv as cv
from src import feature_experiments as fe
from src.features import (
    CONFIG_C, add_time_features, add_weather_features,
    add_history_features, has_history,
)
from src.model import make_cv_model
from src.evaluation import regression_metrics

Đọc kết quả C30 và kiểm tra danh sách ứng viên khớp với mã hiện tại trước khi lấy tham số RF. Nếu không khớp, dừng để tránh fit từ kết quả cũ.

In [2]:
comparison = cv.load_comparison(project_root)

# Dừng nếu kết quả được tạo từ danh sách ứng viên cũ.
assert json.dumps(
    comparison["protocol"]["candidates"], sort_keys=True
) == json.dumps(cv.candidate_bank(), sort_keys=True), (
    "Kết quả tuning chưa khớp code mới. Restart kernel và chạy lại tuning."
)

record = comparison["records"]["C30"]
MODEL_NAME = "Random Forest"
chosen = next(m for m in record["results"] if m["model"] == MODEL_NAME)

print("Mô hình:", MODEL_NAME)
print("Tham số đã chốt:", chosen["best_params"])

Mô hình: Random Forest
Tham số đã chốt: {'model__n_estimators': 400, 'model__min_samples_leaf': 5, 'model__max_features': 0.33, 'model__max_depth': 16}


### 2. Chuẩn bị train và validation, không đọc target test

Đếm timestamp để xác định mốc chia, sau đó chỉ đọc đến cuối validation.

In [3]:
csv_path = project_root / "data/preprocessed/hourly_unfilled.csv"

n_total = len(pd.read_csv(csv_path, usecols=["date_time"]))
train_stop = n_total * 70 // 100
validation_stop = n_total * 85 // 100

hourly = pd.read_csv(
    csv_path,
    nrows=validation_stop,
    keep_default_na=False,
    na_values={"temp": [""], "rain_1h": [""]},
    float_precision="round_trip",
)
hourly["date_time"] = pd.to_datetime(
    hourly["date_time"], format="%Y-%m-%d %H:%M:%S"
)

featured = add_history_features(
    add_weather_features(add_time_features(hourly))
)

fit = featured.iloc[:train_stop].copy()
score = featured.iloc[train_stop:].copy()

fit = fit.loc[has_history(fit)].copy()
score = score.loc[has_history(score)].copy()

assert fit.date_time.max() < score.date_time.min()

print("Train đủ history:", len(fit))
print("Validation đủ history:", len(score))

Train đủ history: 22503
Validation đủ history: 6050


### 3. Học cách điền thiếu từ train

Dùng lại phương pháp của FE. Lúc này fit_times là toàn bộ cohort train, còn score_times là cohort validation. Median chỉ học từ train. Không gọi fit_weather_fill trên validation.


In [4]:
X_train, X_validation, fill_statistics = fe.prepare_fold(
    train=hourly,
    featured=featured,
    fit_times=fit.date_time,
    score_times=score.date_time,
)

# Cột dẫn xuất cho Ridge; RF chỉ chọn đúng CONFIG_C.
for matrix in (X_train, X_validation):
    matrix["weekday_hour"] = (
        matrix.day_of_week * 24 + matrix.hour
    ).astype("int16")

y_train = fit.traffic_volume.to_numpy()
y_validation = score.traffic_volume.to_numpy()

print("Train shape:", X_train.shape, y_train.shape)

assert X_validation.index.equals(pd.DatetimeIndex(score.date_time))

Train shape: (22503, 31) (22503,)


### 4. Fit mô hình bằng tham số đã chọn

In [5]:
model = make_cv_model(MODEL_NAME, CONFIG_C)
model.set_params(**chosen["best_params"])

model.fit(X_train, y_train)

Out[0]: 
Pipeline(steps=[('columns',
                 FeatureColumns(columns=('hour', 'day_of_week', 'month',
                                         'is_weekend', 'is_rush_hour',
                                         'hour_sin', 'hour_cos',
                                         'day_of_week_sin', 'day_of_week_cos',
                                         'month_sin', 'month_cos', 'is_holiday',
                                         'temp_c', 'rain_1h', 'snow_1h',
                                         'clouds_all', 'weather_clear',
                                         'weather_clouds', 'weather_drizzle',
                                         'weather_fog', 'weather_haze',
                                         'weather_mist', 'weather_rain',
                                         'weather_smoke', 'weather_snow',
                                         'weather_squall',
                                         'weather_thunderstorm', 'lag_24h',
                 

### 5. Dự đoán và chấm validation

In [6]:
prediction = np.clip(model.predict(X_validation), 0, None)

metrics = regression_metrics(y_validation, prediction)

validation_result = pd.DataFrame([{
    "model": MODEL_NAME,
    "config": "C30",
    "n": len(y_validation),
    **metrics,
}])

display(validation_result)

,model,config,n,MAE,RMSE,R2
0,Random Forest,C30,6050,200.043773,334.530674,0.970999


## Fit lại sau tuning trên XGBoost

XGBoost dùng lại đúng `X_train`, `X_validation`, `y_train`, `y_validation` và thống kê điền thiếu đã chuẩn bị cho RF. Vì vậy hai mô hình học và chấm trên cùng timestamp. Cột `weekday_hour` là cột phụ cho Ridge; pipeline XGBoost chỉ chọn 30 cột `CONFIG_C`.

### 1. Đọc tham số XGBoost đã chốt

Lấy `best_params` của XGBoost từ kết quả tuning C30, không nhập lại tham số từ ảnh hoặc chọn theo điểm validation.

In [7]:
XGB_MODEL_NAME = "XGBoost"
chosen_xgb = next(m for m in record["results"] if m["model"] == XGB_MODEL_NAME)

print("Mô hình:", XGB_MODEL_NAME)
print("Tham số đã chốt:", chosen_xgb["best_params"])

Mô hình: XGBoost
Tham số đã chốt: {'model__subsample': 1.0, 'model__min_child_weight': 1, 'model__max_depth': 6, 'model__learning_rate': 0.03}


### 2. Fit XGBoost trên toàn bộ cohort train

Tạo pipeline mới và áp dụng tham số đã chọn bằng CV. Lệnh `fit` chỉ nhận train; không dùng validation để early stopping hoặc tìm tham số.

In [8]:
xgb_model = make_cv_model(XGB_MODEL_NAME, CONFIG_C)
xgb_model.set_params(**chosen_xgb["best_params"])

xgb_model.fit(X_train, y_train)

Out[0]: 
Pipeline(steps=[('columns',
                 FeatureColumns(columns=('hour', 'day_of_week', 'month',
                                         'is_weekend', 'is_rush_hour',
                                         'hour_sin', 'hour_cos',
                                         'day_of_week_sin', 'day_of_week_cos',
                                         'month_sin', 'month_cos', 'is_holiday',
                                         'temp_c', 'rain_1h', 'snow_1h',
                                         'clouds_all', 'weather_clear',
                                         'weather_clouds', 'weather_drizzle',
                                         'weather_fog', 'weather_haze',
                                         'weather_mist', 'weather_r...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_co

### 3. Dự đoán và chấm trên cùng validation

Cắt dự đoán âm về 0 bằng cùng quy tắc của RF và CV. Tính MAE, RMSE, R² trên đúng thứ tự `y_validation`; không thay tập giờ để làm điểm tốt hơn.

In [9]:
xgb_prediction = np.clip(xgb_model.predict(X_validation), 0, None)
assert len(xgb_prediction) == len(prediction) == len(y_validation)

xgb_metrics = regression_metrics(y_validation, xgb_prediction)
xgb_validation_result = pd.DataFrame([{
    "model": XGB_MODEL_NAME,
    "config": "C30",
    "n": len(y_validation),
    **xgb_metrics,
}])

display(xgb_validation_result)

,model,config,n,MAE,RMSE,R2
0,XGBoost,C30,6050,201.846339,322.440935,0.973058


### 4. So sánh với RF và nhận xét ngắn

Gộp hai kết quả trên cùng số giờ chấm. Nhận xét dưới đây được tính từ kết quả chạy, không nhập tay số liệu. Đây là đánh giá validation phục vụ lựa chọn; chưa phải điểm test cuối cùng.

In [10]:
validation_comparison = pd.concat(
    [validation_result, xgb_validation_result], ignore_index=True
).sort_values("MAE").reset_index(drop=True)
display(validation_comparison)

rf_mae = float(validation_result["MAE"].iloc[0])
xgb_mae = float(xgb_validation_result["MAE"].iloc[0])
delta_mae = xgb_mae - rf_mae
direction = "cao hơn" if delta_mae > 0 else "thấp hơn" if delta_mae < 0 else "bằng"
display(Markdown(
    f"Trên cùng {len(y_validation):,} giờ validation, XGBoost đạt MAE {xgb_mae:.3f} xe/giờ, "
    f"{direction} RF {abs(delta_mae):.3f} xe/giờ. "
    "Kết quả chỉ áp dụng cho validation này; chưa đánh giá test."
))

Trên cùng 6,050 giờ validation, XGBoost đạt MAE 201.846 xe/giờ, cao hơn RF 1.803 xe/giờ. Kết quả chỉ áp dụng cho validation này; chưa đánh giá test.